# GShell — Debug Playground

Step-through notebook for `train_gshelltet_polycam.py`.  
Each cell mirrors one stage of the training script so you can set breakpoints, inspect tensors, and iterate.

## 0 — Environment & cache setup

In [2]:
import os

cache_root = "/data/abelde"
repo_root = "/data/abelde/projects/active/Shell_Gaussian/baselines/GShell"
env_root = os.path.join(repo_root, "GShell_env")
env_bin = os.path.join(env_root, "bin")
conda_gcc = os.path.join(env_bin, "x86_64-conda-linux-gnu-gcc")
conda_gxx = os.path.join(env_bin, "x86_64-conda-linux-gnu-g++")
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
os.environ["PATH"] = env_bin + os.pathsep + os.environ.get("PATH", "")
os.environ.setdefault("CUDA_HOME", env_root)
os.environ.setdefault("CUDA_PATH", env_root)
os.environ["CC"] = conda_gcc
os.environ["CXX"] = conda_gxx
os.environ["CUDAHOSTCXX"] = conda_gxx

os.environ["PIP_CACHE_DIR"]  = os.path.join(cache_root, ".cache", "pip")
os.environ["TORCH_HOME"]     = os.path.join(cache_root, ".cache", "torch")
os.environ["HF_HOME"]        = os.path.join(cache_root, ".cache", "huggingface")
os.environ["XDG_CACHE_HOME"] = os.path.join(cache_root, ".cache")
os.environ["TMPDIR"]         = os.path.join(cache_root, "tmp")

lib_prefixes = ["/usr/lib/x86_64-linux-gnu", os.path.join(env_root, "lib"), os.path.join(env_root, "lib64")]
for key in ["LIBRARY_PATH", "LD_LIBRARY_PATH"]:
    existing = os.environ.get(key, "")
    os.environ[key] = os.pathsep.join(lib_prefixes + ([existing] if existing else []))

for d in [os.environ["PIP_CACHE_DIR"], os.environ["TORCH_HOME"],
          os.environ["HF_HOME"], os.environ["XDG_CACHE_HOME"], os.environ["TMPDIR"]]:
    os.makedirs(d, exist_ok=True)

print("Cache directories set under:", cache_root)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])
print("Env bin on PATH:", env_bin)
print("CC:", os.environ["CC"], "exists=", os.path.exists(os.environ["CC"]))
print("CXX:", os.environ["CXX"], "exists=", os.path.exists(os.environ["CXX"]))

Cache directories set under: /data/abelde
CUDA_VISIBLE_DEVICES: 6
Env bin on PATH: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin
CC: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/x86_64-conda-linux-gnu-gcc exists= True
CXX: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/x86_64-conda-linux-gnu-g++ exists= True


## 1 — Imports & paths

In [3]:
import os, sys, time, json, argparse
import numpy as np

repo_root = "/data/abelde/projects/active/Shell_Gaussian/baselines/GShell"
project_root = "/data/abelde/projects/active/Shell_Gaussian"
foot_root = os.path.join(project_root, "FootShellGaussian")

foreign_modules = []
for name, module in list(sys.modules.items()):
    module_file = getattr(module, "__file__", "") or ""
    if module_file.startswith(foot_root):
        foreign_modules.append((name, module_file))
if foreign_modules:
    sample = "\n".join(f"  {name}: {path}" for name, path in foreign_modules[:5])
    raise RuntimeError(
        "This kernel already imported FootShellGaussian modules. Restart the kernel, "
        "then run from the top so the GShell modules and C++ extensions load cleanly.\n" + sample
    )

for path in (repo_root, foot_root):
    while path in sys.path:
        sys.path.remove(path)
sys.path.insert(0, repo_root)
os.chdir(repo_root)

print("Warming up torch...")
t0 = time.time()
import torch
print(f"torch imported in {time.time() - t0:.1f}s  |  version: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Logical cuda:0 maps to physical GPU:", os.environ.get("CUDA_VISIBLE_DEVICES", "all visible GPUs"))

import nvdiffrast.torch as dr
import xatlas

from dataset.dataset_nerf_colmap import DatasetNERF
from geometry.gshell_tets_geometry import GShellTetsGeometry
from render import renderutils as ru
from render import obj, material, mesh, texture, mlptexture, light, render, util
from denoiser.denoiser import BilateralDenoiser

print("All imports OK")

Warming up torch...
torch imported in 1.2s  |  version: 1.13.1  |  CUDA: True
GPU: NVIDIA A100-SXM4-80GB
Logical cuda:0 maps to physical GPU: 6


Detected CUDA files, patching ldflags
Emitting ninja build file /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/render/optixutils/build/build.ninja...
Building extension module optixutils_plugin...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)
Loading extension module optixutils_plugin...


ninja: no work to do.
All imports OK


## 2 — Configuration (FLAGS)

Uses `smoke_test.json` by default for fast iteration. Switch to `shoes_mc.json` for full-quality runs.  
Edit `config_path` and `data_path` to point to your dataset.

In [4]:
# ── Knobs: change these for your run ──
config_path = os.path.join(repo_root, "configs/shoes_mc_normfix.json")   # fast debug; swap to shoes_mc.json for real runs
data_path   = "/data/abelde/datasets/processed/gshell_shoes/Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Kids"
output_dir  = os.path.join(repo_root, "output/debug_playground")

os.makedirs(output_dir, exist_ok=True)

# ── Build FLAGS exactly as train_gshelltet_polycam.py does ──
parser = argparse.ArgumentParser(description="nvdiffrec")
parser.add_argument("--config", type=str, default=None)
parser.add_argument("-i", "--iter", type=int, default=5000)
parser.add_argument("-b", "--batch", type=int, default=1)
parser.add_argument("-s", "--spp", type=int, default=1)
parser.add_argument("-l", "--layers", type=int, default=1)
parser.add_argument("-r", "--train-res", nargs=2, type=int, default=[512, 512])
parser.add_argument("-dr", "--display-res", type=int, default=None)
parser.add_argument("-tr", "--texture-res", nargs=2, type=int, default=[1024, 1024])
parser.add_argument("-di", "--display-interval", type=int, default=0)
parser.add_argument("-si", "--save-interval", type=int, default=1000)
parser.add_argument("-lr", "--learning-rate", type=float, default=0.01)
parser.add_argument("-mr", "--min-roughness", type=float, default=0.08)
parser.add_argument("-mip", "--custom-mip", action="store_true", default=False)
parser.add_argument("-rt", "--random-textures", action="store_true", default=False)
parser.add_argument("-bg", "--background", default="checker", choices=["black", "white", "checker", "reference"])
parser.add_argument("--loss", default="logl1", choices=["logl1", "logl2", "mse", "smape", "relmse"])
parser.add_argument("-o", "--out-dir", type=str, default=None)
parser.add_argument("-rm", "--ref_mesh", type=str)
parser.add_argument("-bm", "--base-mesh", type=str, default=None)
parser.add_argument("--validate", type=bool, default=True)
parser.add_argument("--n_samples", type=int, default=4)
parser.add_argument("--bsdf", type=str, default="pbr", choices=["pbr", "diffuse", "white"])
parser.add_argument("--denoiser", default="bilateral", choices=["none", "bilateral"])
parser.add_argument("--denoiser_demodulate", type=bool, default=True)
parser.add_argument("--msdf_reg_open_scale", type=float, default=1e-6)
parser.add_argument("--msdf_reg_close_scale", type=float, default=3e-4)
parser.add_argument("--eikonal_scale", type=float, default=5e-3)
parser.add_argument("--sdf_regularizer", type=float, default=0.2)
parser.add_argument("--trainset_path", type=str)
parser.add_argument("--testset_path", type=str, default="")

FLAGS = parser.parse_args([
    "--config",        config_path,
    "--trainset_path", data_path,
    "--out-dir",       output_dir,
])

# ── Hardcoded defaults from the training script ──
FLAGS.mtl_override                = None
'''
Think of the 3D reconstruction space as being divided into little cells, like a Minecraft world or a 3D grid.

A low grid resolution means:
fewer cells
coarser geometry
faster training
less detail

A high grid resolution means:
more cells
more detailed geometry
slower training
more memory
'''
FLAGS.gshell_grid                 = 64. # This controls the resolution of the geometry scaffold. 
FLAGS.mesh_scale                  = 1.0
FLAGS.envlight                    = None
FLAGS.env_scale                   = 1.0
FLAGS.probe_res                   = 256
FLAGS.learn_lighting              = True
FLAGS.display                     = None
FLAGS.transparency                = False
FLAGS.lock_light                  = False
FLAGS.lock_pos                    = False
FLAGS.laplace                     = "relative"
FLAGS.laplace_scale               = 3000.0
FLAGS.pre_load                    = True
FLAGS.no_perturbed_nrm            = False
FLAGS.decorrelated                = False
FLAGS.kd_min                      = [0.0, 0.0, 0.0, 0.0]
FLAGS.kd_max                      = [1.0, 1.0, 1.0, 1.0]
FLAGS.ks_min                      = [0.0, 0.001, 0.0]
FLAGS.ks_max                      = [0.0, 1.0, 1.0]
FLAGS.nrm_min                     = [-1.0, -1.0, 0.0]
FLAGS.nrm_max                     = [1.0, 1.0, 1.0]
FLAGS.clip_max_norm               = 0.0
FLAGS.cam_near_far                = [0.1, 1000.0]
FLAGS.lambda_kd                   = 0.1
FLAGS.lambda_ks                   = 0.05
FLAGS.lambda_nrm                  = 0.025
FLAGS.lambda_nrm2                 = 0.25
FLAGS.lambda_chroma               = 0.0
FLAGS.lambda_diffuse              = 0.15
FLAGS.lambda_specular             = 0.0025
FLAGS.random_lgt                  = False
FLAGS.normal_only                 = False
FLAGS.use_img_2nd_layer           = False
FLAGS.use_depth                   = False
FLAGS.use_depth_2nd_layer         = False
FLAGS.use_tanh_deform             = False
FLAGS.use_sdf_mlp                 = True
FLAGS.use_msdf_mlp                = False
FLAGS.use_eikonal                 = True
FLAGS.sdf_mlp_pretrain_steps      = 4000
FLAGS.use_mesh_msdf_reg           = True
FLAGS.sphere_init                 = False
FLAGS.sphere_init_norm            = 0.17
FLAGS.pretrained_sdf_mlp_path     = f"./data/pretrained_mlp_{FLAGS.gshell_grid}_polycam.pt"
FLAGS.n_hidden                    = 6
FLAGS.d_hidden                    = 256
FLAGS.n_freq                      = 6
FLAGS.skip_in                     = [3]
FLAGS.use_float16                 = False
FLAGS.visualize_watertight        = False
FLAGS.local_rank                  = 0
FLAGS.multi_gpu                   = False

# ── Apply JSON config overrides ──
if FLAGS.config is not None:
    with open(FLAGS.config, "r") as f:
        for key, val in json.load(f).items():
            setattr(FLAGS, key, val)

if FLAGS.display_res is None:
    FLAGS.display_res = FLAGS.train_res

print("Config / Flags:")
print("-" * 40)
for k, v in sorted(vars(FLAGS).items()):
    print(f"  {k}: {v}")
print("-" * 40)

Config / Flags:
----------------------------------------
  aabb: [-1, -1, -1, 1, 1, 1]
  background: white
  base_mesh: None
  batch: 2
  boxscale: [1, 1, 1]
  bsdf: pbr
  cam_near_far: [0.1, 1000.0]
  clip_max_norm: 0.0
  config: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/configs/shoes_mc_normfix.json
  custom_mip: False
  d_hidden: 256
  decorrelated: False
  denoiser: bilateral
  denoiser_demodulate: True
  display: [{'latlong': True}]
  display_interval: 0
  display_res: [1800, 2400]
  eikonal_scale: 0.005
  env_scale: 2.0
  envlight: data/irrmaps/aerodynamics_workshop_2k.hdr
  gshell_grid: 128
  iter: 5000
  kd_max: [1.0, 1.0, 1.0, 1.0]
  kd_min: [0.0, 0.0, 0.0, 0.0]
  ks_max: [0, 1.0, 1.0]
  ks_min: [0, 0.001, 0.0]
  lambda_chroma: 0.0
  lambda_diffuse: 0.15
  lambda_kd: 0.1
  lambda_ks: 0.05
  lambda_nrm: 0.025
  lambda_nrm2: 0.25
  lambda_specular: 0.0025
  laplace: relative
  laplace_scale: 6000
  layers: 1
  learn_lighting: True
  learning_rate: [0.0025, 0.0

## 3 — Dataset

In [5]:
data_root = FLAGS.trainset_path

dataset_train    = DatasetNERF(os.path.join(data_root, "transforms.json"), FLAGS, examples=int(1e6))
dataset_validate = DatasetNERF(os.path.join(data_root, "transforms.json"), FLAGS)

print(f"Train examples : {len(dataset_train)}")
print(f"Val   examples : {len(dataset_validate)}")

sample = dataset_train[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:12s}  {str(v.shape):20s}  {v.dtype}")
    else:
        print(f"  {k:12s}  {type(v).__name__}: {v}")

DatasetNERF: 36 images with shape [1800, 2400]
DatasetNERF: 36 images with shape [1800, 2400]
Train examples : 1000000
Val   examples : 36
  mv            torch.Size([1, 4, 4])  torch.float32
  mvp           torch.Size([1, 4, 4])  torch.float32
  campos        torch.Size([1, 3])    torch.float32
  resolution    list: [1800, 2400]
  spp           int: 1
  img           torch.Size([1, 1800, 2400, 4])  torch.float32


## 4 — Rasterization context, lighting, denoiser

In [6]:
glctx = dr.RasterizeGLContext()

if FLAGS.learn_lighting:
    lgt = light.create_trainable_env_rnd(FLAGS.probe_res, scale=0.0, bias=0.5)
else:
    lgt = light.load_env(FLAGS.envlight, scale=FLAGS.env_scale, res=[FLAGS.probe_res, FLAGS.probe_res])

denoiser = None
if FLAGS.denoiser == "bilateral":
    denoiser = BilateralDenoiser().cuda()

print(f"Light type   : {type(lgt).__name__}")
print(f"Denoiser     : {type(denoiser).__name__ if denoiser else 'None'}")

Light type   : EnvironmentLight
Denoiser     : BilateralDenoiser


/tmp/ipykernel_989702/482287482.py:1: DeprecationWarning: RasterizeGLContext has been deprecated and uses RasterizeCudaContext internally
  glctx = dr.RasterizeGLContext()
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/lib/python3.10/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /opt/conda/conda-bld/pytorch_1670525541990/work/aten/src/ATen/native/TensorShape.cpp:3190.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


## 5 — Geometry & material initialization

In [7]:
geometry = GShellTetsGeometry(FLAGS.gshell_grid, FLAGS.mesh_scale, FLAGS)

print(f"Tet grid res : {FLAGS.gshell_grid}")
print(f"Mesh scale   : {FLAGS.mesh_scale}")
print(f"Verts        : {geometry.verts.shape}")
print(f"Indices      : {geometry.indices.shape}")
print(f"Parameters   :")
for name, param in geometry.named_parameters():
    print(f"  {name:40s}  {str(list(param.shape)):20s}  requires_grad={param.requires_grad}")

Cuda path /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env
End of OptiXStateWrapper 
using resolution 128


100%|██████████| 4000/4000 [03:18<00:00, 20.10it/s]


sdf net trained with loss: tensor(4.0942e-07, device='cuda:0', grad_fn=<MeanBackward0>)
Tet grid res : 128
Mesh scale   : 1.0
Verts        : torch.Size([277410, 3])
Indices      : torch.Size([1524684, 4])
Parameters   :
  sdf                                       [277410]              requires_grad=True
  msdf                                      [277410]              requires_grad=True
  deform                                    [277410, 3]           requires_grad=True
  sdf_net.net.0.weight                      [256, 39]             requires_grad=True
  sdf_net.net.0.bias                        [256]                 requires_grad=True
  sdf_net.net.2.weight                      [256, 256]            requires_grad=True
  sdf_net.net.2.bias                        [256]                 requires_grad=True
  sdf_net.net.4.weight                      [256, 256]            requires_grad=True
  sdf_net.net.4.bias                        [256]                 requires_grad=True
  sdf_net.net.6

In [10]:
from train_gshelltet_polycam import initial_guess_material

mat = initial_guess_material(geometry, True, FLAGS, None)
mat["no_perturbed_nrm"] = True

print(f"Material keys : {list(mat.keys())}")
if "kd_ks" in mat:
    print(f"  kd_ks type  : {type(mat['kd_ks']).__name__}")
    for n, p in mat["kd_ks"].named_parameters():
        print(f"    {n:30s}  {str(list(p.shape)):20s}")

Encoder output: 32 dims
Material keys : ['kd_ks', 'bsdf', 'no_perturbed_nrm']
  kd_ks type  : MLPTexture3D
    encoder.params                  [12599920]          
    net.net.0.weight                [32, 32]            
    net.net.2.weight                [32, 32]            
    net.net.4.weight                [6, 32]             


## 6 — Optimizers & LR schedule

In [11]:
from train_gshelltet_polycam import createLoss

def resolve_learning_rates(value):
    if isinstance(value, (list, tuple)):
        if len(value) > 0 and isinstance(value[0], (list, tuple)):
            value = value[0]
        learning_rate_pos = float(value[0])
        learning_rate_mat = float(value[1]) if len(value) > 1 else learning_rate_pos
        learning_rate_lgt = float(value[2]) if len(value) > 2 else learning_rate_pos * 6.0
        return learning_rate_pos, learning_rate_mat, learning_rate_lgt
    value = float(value)
    return value, value, value * 6.0

learning_rate_pos, learning_rate_mat, learning_rate_lgt = resolve_learning_rates(FLAGS.learning_rate)
print(f"LR pos={learning_rate_pos}, mat={learning_rate_mat}, lgt={learning_rate_lgt}")

def lr_schedule(iter, fraction):
    return max(0.0, 10 ** (-iter * 0.0002))

image_loss_fn = createLoss(FLAGS)

# Material optimizer
params = list(material.get_parameters(mat))
optimizer = torch.optim.Adam(params, lr=learning_rate_mat)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda x: lr_schedule(x, 0.9))

# Light optimizer
optimizer_light = torch.optim.Adam((lgt.parameters() if lgt is not None else []), lr=learning_rate_lgt)
scheduler_light = torch.optim.lr_scheduler.LambdaLR(optimizer_light, lr_lambda=lambda x: lr_schedule(x, 0.9))

# Geometry optimizer
if FLAGS.use_sdf_mlp:
    lr_msdf = learning_rate_pos * 1e-2 if FLAGS.use_msdf_mlp else learning_rate_pos
    deform_params = [v[1] for v in geometry.named_parameters() if "deform" in v[0]]
    msdf_params   = [v[1] for v in geometry.named_parameters() if "msdf" in v[0]]
    sdf_params    = [v[1] for v in geometry.named_parameters() if "sdf" in v[0] and "msdf" not in v[0]]
    other_params  = [v[1] for v in geometry.named_parameters() if "sdf" not in v[0] and "msdf" not in v[0] and "deform" not in v[0]]
    optimizer_mesh = torch.optim.Adam([
        {"params": deform_params, "lr": learning_rate_pos},
        {"params": msdf_params,   "lr": lr_msdf},
        {"params": sdf_params,    "lr": learning_rate_pos * 1e-2},
        {"params": other_params,  "lr": learning_rate_pos * 1e-2},
    ], eps=1e-8)
else:
    optimizer_mesh = torch.optim.Adam(geometry.parameters(), lr=learning_rate_pos)
scheduler_mesh = torch.optim.lr_scheduler.LambdaLR(optimizer_mesh, lr_lambda=lambda x: lr_schedule(x, 0.9))

print(f"Optimizer param groups (mesh): {len(optimizer_mesh.param_groups)}")
print(f"Optimizer param groups (mat) : {len(optimizer.param_groups)}")

LR pos=0.0025, mat=0.005, lgt=0.015
Optimizer param groups (mesh): 4
Optimizer param groups (mat) : 1


## 7 — Single training step (the debugging cell)

This is the core loop from `optimize_mesh`, broken out so you can step through it, inspect `img_loss`, `reg_loss`, gradients, and intermediate mesh states.

In [12]:
from train_gshelltet_polycam import prepare_batch

dataloader_train = torch.utils.data.DataLoader(
    dataset_train, batch_size=FLAGS.batch, collate_fn=dataset_train.collate, shuffle=True
)
train_iter = iter(dataloader_train)

# ── Fetch one batch ──
target = next(train_iter)
target = prepare_batch(target, "random")

print("Batch keys:", list(target.keys()))
print(f"  img       : {target['img'].shape}  device={target['img'].device}")
print(f"  mv        : {target['mv'].shape}")
print(f"  mvp       : {target['mvp'].shape}")
print(f"  campos    : {target['campos'].shape}")
print(f"  background: {target['background'].shape}")

Batch keys: ['mv', 'mvp', 'campos', 'resolution', 'spp', 'img', 'img_second', 'invdepth', 'invdepth_second', 'envlight_transform', 'background']
  img       : torch.Size([2, 1800, 2400, 4])  device=cuda:0
  mv        : torch.Size([2, 4, 4])
  mvp       : torch.Size([2, 4, 4])
  campos    : torch.Size([2, 3])
  background: torch.Size([2, 1800, 2400, 3])


In [13]:
# ── Forward pass ──
optimizer.zero_grad()
optimizer_mesh.zero_grad()
optimizer_light.zero_grad()

if lgt is not None:
    lgt.update_pdf()

it = 0  # iteration counter — change for debugging specific iteration behavior
img_loss, depth_loss, reg_loss = geometry.tick(
    glctx, target, lgt, mat, image_loss_fn, it, denoiser=denoiser
)

total_loss = img_loss + reg_loss

print(f"img_loss   = {img_loss.item():.6f}")
print(f"depth_loss = {depth_loss.item():.6f}")
print(f"reg_loss   = {reg_loss.item():.6f}")
print(f"total_loss = {total_loss.item():.6f}")

RuntimeError: Ninja is required to load C++ extensions

In [ ]:
# ── Backward + optimizer step ──
total_loss.backward()

if hasattr(lgt, "base") and lgt.base.grad is not None:
    lgt.base.grad *= 64
if "kd_ks" in mat:
    mat["kd_ks"].encoder.params.grad /= 8.0

optimizer.step()
scheduler.step()
optimizer_mesh.step()
scheduler_mesh.step()
optimizer_light.step()
scheduler_light.step()

# ── Clamping ──
with torch.no_grad():
    if "kd" in mat:
        mat["kd"].clamp_()
    if "ks" in mat:
        mat["ks"].clamp_()
    if lgt is not None:
        lgt.clamp_(min=1e-4)
    geometry.clamp_deform()

torch.cuda.current_stream().synchronize()
print("Step completed. Inspect geometry / mat / lgt freely below.")

## 8 — Multi-step training loop

Run N iterations to watch convergence. Re-run this cell to continue from where you left off.

In [ ]:
import tqdm

N_STEPS = FLAGS.iter  # change to a smaller number for quick tests
log_interval = 10

img_loss_vec = []
reg_loss_vec = []

dataloader = torch.utils.data.DataLoader(
    dataset_train, batch_size=FLAGS.batch, collate_fn=dataset_train.collate, shuffle=True
)

for it, target in enumerate(tqdm.tqdm(dataloader, total=min(N_STEPS, len(dataloader)))):
    target = prepare_batch(target, "random")

    optimizer.zero_grad()
    optimizer_mesh.zero_grad()
    optimizer_light.zero_grad()

    if lgt is not None:
        lgt.update_pdf()

    try:
        img_loss, depth_loss, reg_loss = geometry.tick(
            glctx, target, lgt, mat, image_loss_fn, it, denoiser=denoiser
        )
    except render.EmptyMeshError:
        optimizer.zero_grad()
        optimizer_mesh.zero_grad()
        optimizer_light.zero_grad()
        print(f"[iter={it}] Empty mesh — skipping")
        continue

    total_loss = img_loss + reg_loss
    total_loss.backward()

    if hasattr(lgt, "base") and lgt.base.grad is not None:
        lgt.base.grad *= 64
    if "kd_ks" in mat:
        mat["kd_ks"].encoder.params.grad /= 8.0

    optimizer.step();       scheduler.step()
    optimizer_mesh.step();  scheduler_mesh.step()
    optimizer_light.step(); scheduler_light.step()

    with torch.no_grad():
        if "kd" in mat: mat["kd"].clamp_()
        if "ks" in mat: mat["ks"].clamp_()
        if lgt is not None: lgt.clamp_(min=1e-4)
        geometry.clamp_deform()

    img_loss_vec.append(img_loss.item())
    reg_loss_vec.append(reg_loss.item())

    if it % log_interval == 0:
        avg_img = np.mean(img_loss_vec[-log_interval:])
        avg_reg = np.mean(reg_loss_vec[-log_interval:])
        tqdm.tqdm.write(f"iter={it:5d}  img_loss={avg_img:.6f}  reg_loss={avg_reg:.6f}  lr={optimizer.param_groups[0]['lr']:.5f}")

    if it >= N_STEPS:
        break

torch.cuda.current_stream().synchronize()
print(f"\nFinished {min(it+1, N_STEPS)} iterations.")

## 9 — Loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(img_loss_vec, linewidth=0.8)
ax1.set_title("Image loss")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Loss")
ax1.set_yscale("log")

ax2.plot(reg_loss_vec, linewidth=0.8, color="tab:orange")
ax2.set_title("Regularization loss")
ax2.set_xlabel("Iteration")
ax2.set_ylabel("Loss")
ax2.set_yscale("log")

plt.tight_layout()
plt.show()

## 10 — Extract & inspect mesh

In [ ]:
with torch.no_grad():
    result = geometry.getMesh(mat)
    base_mesh = result["imesh"]

    print(f"Vertices  : {base_mesh.v_pos.shape}")
    print(f"Triangles : {base_mesh.t_pos_idx.shape}")
    print(f"v_pos range: min={base_mesh.v_pos.min(dim=0).values.cpu().numpy()}, "
          f"max={base_mesh.v_pos.max(dim=0).values.cpu().numpy()}")

    if hasattr(base_mesh, "v_nrm") and base_mesh.v_nrm is not None:
        print(f"Normals   : {base_mesh.v_nrm.shape}")

## 11 — Render a validation view & compare to GT

In [ ]:
from train_gshelltet_polycam import validate_itr

dataloader_validate = torch.utils.data.DataLoader(
    dataset_validate, batch_size=1, collate_fn=dataset_validate.collate
)

val_target = next(iter(dataloader_validate))
val_target = prepare_batch(val_target, FLAGS.background)

result_image, result_dict = validate_itr(glctx, val_target, geometry, mat, lgt, FLAGS, denoiser=denoiser)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

opt_img = result_dict["opt"].detach().cpu().numpy()
ref_img = result_dict["ref"].detach().cpu().numpy()

axes[0].imshow(np.clip(opt_img, 0, 1))
axes[0].set_title("Rendered")
axes[0].axis("off")

axes[1].imshow(np.clip(ref_img, 0, 1))
axes[1].set_title("Ground Truth")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 12 — Save outputs (mesh OBJ + model checkpoint)

In [ ]:
with torch.no_grad():
    mesh_dir = os.path.join(output_dir, "mesh")
    os.makedirs(mesh_dir, exist_ok=True)

    # Save model weights
    torch.save(geometry.state_dict(), os.path.join(mesh_dir, "model.pt"))
    torch.save(mat["kd_ks"].state_dict(), os.path.join(mesh_dir, "mtl.pt"))
    light.save_env_map(os.path.join(mesh_dir, "probe.hdr"), lgt)

    # Export OBJ
    export_mesh = geometry.getMesh(mat)["imesh"]
    obj.write_obj(os.path.join(mesh_dir, ""), export_mesh, save_material=False)

    print(f"Saved to {mesh_dir}/")
    for f in sorted(os.listdir(mesh_dir)):
        fpath = os.path.join(mesh_dir, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f:30s}  {size_mb:.2f} MB")

## 13 — Scratch cell

Use this for ad-hoc inspection: gradient norms, SDF values, parameter histograms, etc.

In [ ]:
# Example: inspect gradient norms per parameter group
for name, param in geometry.named_parameters():
    grad_norm = param.grad.norm().item() if param.grad is not None else 0.0
    print(f"  {name:40s}  val_norm={param.data.norm().item():.4f}  grad_norm={grad_norm:.6f}")